<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_03_target_definition/stage_03b_target_definition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03b_target_definition**


## **Configuración del Entorno**


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

from __future__ import annotations

# ---------------------------------------------------------------------
# Imports (stdlib)
# ---------------------------------------------------------------------
import argparse
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

# ---------------------------------------------------------------------
# Imports (third-party)
# ---------------------------------------------------------------------
import numpy as np
import pandas as pd
from numba import njit

# ---------------------------------------------------------------------
# Logging (uniforme)
# ---------------------------------------------------------------------
import logging

logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_03a")

### 0.3. Definición de rutas

In [3]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [4]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03a_target_investigation_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_03b_target_definition_summary.json"))


In [5]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
#OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## **1. Cierre de investigación de objetivos**

En la notebook `stage_03a_target_investigation` se realizó una investigación empírica sobre el comportamiento intradía del MNQ con el objetivo de:

- Identificar ventanas temporales óptimas de operación.
- Estimar deltas económicamente relevantes (base, operativo y cola).
- Y derivar stop loss empíricos basados en el MAE observado hasta el alcance del target.

El análisis se efectuó para horizontes de 60 y 90 minutos, evaluando persistencia temporal, distribución de magnitudes y relaciones riesgo–beneficio.

Los resultados muestran que ambos horizontes son operables, con:

- Ventanas tempranas bien definidas (≈09:10–09:40).
- Targets p70 claramente superiores al delta base.
- Y risk–reward recomendados estables en torno a 2.2.

Los resultados de la investigación son:

## **2. Definición de parámetros operativos**

En base a los resultados obtenidos, se fijan los siguientes parámetros operativos de referencia, que serán utilizados para la generación de targets, stops y etiquetas en el pipeline.

### **2.1. Horizonte operativo principal**

- Horizontes evaluados: 60 min y 90 min
- Horizonte preferente: 90 minutos
  - Mayor estabilidad temporal
  - Ventana ligeramente más amplia
  - Risk–reward marginalmente superior

  El horizonte de 60 minutos se mantiene como referencia secundaria y comparativa.

### **2.2. Ventana horaria de operación**

| Horizonte | Inicio | Fin  |
|-----------|--------|------|
| 60 min    | 09:14  | 09:41 |
| 90 min    | 09:09  | 09:36 |

Estas ventanas definen el bloque temporal elegible para señales y evaluación de trades.

### **2.3. Umbrales operativos en puntos (MNQ)**

| Horizonte | Delta base (mediana) | Target operativo (p70) | Cola (p90) |
|-----------|---------------------|------------------------|-----------|
| 60 min    | 52.12 pts           | 84.18 pts              | 140.70 pts|
| 90 min    | 60.75 pts           | 97.22 pts              | 162.30 pts|

- El target p70 se adopta como take profit operativo estándar.
- El p90 se reserva para análisis de colas y escenarios expansivos.

### **2.4. Stop Loss operativo (empiríco)**


| Horizonte | SL p70 | SL p80 |
|-----------|----------------------|--------|
| 60 min    | 37.84 pts            | 49.80 pts |
| 90 min    | 43.42 pts            | 56.75 pts |

- El SL p70 se adopta como stop loss operativo base.
- Derivado directamente del MAE hasta el alcance del TP.

### **2.5. Relación riesgo-beneficio (R/R: Risk/Reward) adoptada**



Se establece como criterio operativo mínimo un riesgo–beneficio de $𝑅𝑅 ≥ 2.0$

| Horizonte | RR (TP p70 / SL p70) |
|-----------|----------------------------------|
| 60 min    | 2.22 |
| 90 min    | 2.24 |

En función de este umbral, se adopta el RR definido como $TP_{𝑝70} / SL_{𝑝70}$, por tratarse de una relación riesgo–retorno consistente con el perfil operativo buscado y respaldada empíricamente por los resultados obtenidos.


### **2.6. Resumen final**


Con esta definición:

- Se congela la investigación empírica (03a),
- Se fijan parámetros económicos explícitos,
- Se habilita la generación consistente de targets y etiquetas.

In [6]:
# Helper: obtener fila por horizonte
def _get_row_by_horizon(data: dict, h: int) -> dict:
    return next(
        row for row in data["details"]["summary_rows"]
        if row["horizon_min"] == h
    )

In [ ]:
# Horizonte 60
row_60 = _get_row_by_horizon(stage_03a_summary, 60)

delta_base_60 = row_60["delta_base_med"]
delta_op_60   = row_60["delta_target_p70"]
delta_tail_60 = row_60["delta_tail_p90"]

# Horizonte 90
row_90 = _get_row_by_horizon(stage_03a_summary, 90)

delta_base_90 = row_90["delta_base_med"]
delta_op_90   = row_90["delta_target_p70"]
delta_tail_90 = row_90["delta_tail_p90"]

## **3. Evaluación de feature de entrada `close`**




### 3.1. Justificación de análisis

A partir de la definición de los parámetros de operación establecidos en el punto anterior, resulta necesario analizar la variable `close`, la cual representa el nivel del índice MNQ en cada instante temporal del dataset intradía.

La evaluación de esta variable constituye un paso previo e indispensable al análisis estadístico de los retornos, por los siguientes motivos:

---

**1. Relación entre objetivos económicos y nivel del índice**

Los objetivos definidos en el Punto 2 se expresan en puntos del índice. Sin embargo, los retornos utilizados en el modelado se calcularán en forma logarítmica, según la expresión:

$$
    r_t = \ln\left( \frac{close_{t+h}}{close_t} \right)
$$

Esto implica que un mismo desplazamiento en puntos absolutos del índice no se traduce en un retorno constante, sino que depende directamente del valor de `close_t`. Por lo tanto, para poder relacionar correctamente los objetivos económicos definidos en puntos con los retornos logarítmicos, es imprescindible conocer el orden de magnitud y la variabilidad del nivel del índice.

---

**2. Justificación del uso de close como referencia de escala**

La variable `close` actúa como factor de escala entre el retorno logarítmico y la variación absoluta en puntos del MNQ. En términos prácticos, la conversión puede aproximarse como:

$$
\Delta \text{puntos} \approx r_t \times close_t
$$

En consecuencia:

- definir umbrales de retorno sin considerar `close` conduce a criterios arbitrarios

- analizar `close` permite establecer umbrales de retorno dinámicos, coherentes con distintos niveles del índice.

---

**3. Necesidad de una referencia común antes del análisis estadístico**

Dado que el dataset abarca múltiples períodos y regímenes de mercado, el nivel del índice MNQ presenta variaciones significativas a lo largo del tiempo. Por este motivo, antes de evaluar la frecuencia y distribución de los retornos, es metodológicamente correcto:

  1. Comprender la escala real del índice representada por `close`.
  2. Validar que los objetivos definidos en puntos sean razonables en todo el período analizado.
  3. Evitar sesgos derivados de asumir un nivel de precios constante.

---

**4 Rol de este paso dentro del pipeline**

La evaluación de la variable `close` cumple, dentro del pipeline, la función de:

- Conectar los objetivos económicos con las variables financieras del dataset.
- Establecer una base sólida para la posterior evaluación estadística de los retornos.
- Garantizar coherencia entre la formulación del problema y la realidad operativa.

---

En síntesis, el análisis de la feature `close` no persigue inicialmente fines estadísticos, sino que constituye un paso conceptual y metodológico orientado a asegurar que los objetivos económicos definidos sean correctamente traducidos al lenguaje de los retornos financieros. Solo a partir de esta validación resulta pertinente avanzar al análisis estadístico de los retornos y a la definición final de los targets de predicción.

### 3.2. Aplicación de análisis

#### 2.2.1. Carga de dataset `intraday_mnq`


In [8]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [9]:
def add_column_date(df):
    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [10]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [11]:
mnq_intraday = load_mnq_parquet()
mnq_intraday = add_column_date(mnq_intraday)
mnq_intraday.head()


Archivo encontrado en disco. Cargando dataset local...


,date,open,high,low,close,volume
datetime,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3


In [12]:
info_dataset(mnq_intraday)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York


#### 2.2.2. Análisis estadistico de `close`


In [13]:
import pandas as pd
from typing import Dict, Tuple

def analyze_close_statistics(
    df: pd.DataFrame,
    close_col: str = "close",
    percentiles=(0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99),
    by_year: bool = True
) -> Tuple[pd.Series, Dict[str, float], pd.DataFrame | None]:
    """
    Calcula estadísticas descriptivas de la variable close.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con índice datetime y columna close.
    close_col : str
        Nombre de la columna de precios (default: 'close').
    percentiles : tuple
        Percentiles a calcular.
    by_year : bool
        Si True, devuelve estadísticas agregadas por año.

    Retorna
    -------
    close_describe : pd.Series
        Resultado de df[close].describe(percentiles=...)
    summary : dict
        Resumen compacto con métricas clave.
    close_by_year : pd.DataFrame | None
        Estadísticas por año (o None si by_year=False).
    """

    close_series = df[close_col].dropna()

    # 1) Estadísticas globales
    close_describe = close_series.describe(percentiles=percentiles)

    # 2) Resumen compacto
    summary = {
        "count": int(close_series.count()),
        "mean": float(close_series.mean()),
        "std": float(close_series.std()),
        "min": float(close_series.min()),
        "p01": float(close_series.quantile(0.01)),
        "p05": float(close_series.quantile(0.05)),
        "p50": float(close_series.quantile(0.50)),
        "p95": float(close_series.quantile(0.95)),
        "p99": float(close_series.quantile(0.99)),
        "max": float(close_series.max()),
    }

    # 3) Estadísticas por año (opcional)
    close_by_year = None
    if by_year:
        tmp = df[[close_col]].copy()
        tmp["year"] = tmp.index.year

        close_by_year = tmp.groupby("year")[close_col].agg(
            count="count",
            mean="mean",
            median="median",
            p05=lambda s: s.quantile(0.05),
            p95=lambda s: s.quantile(0.95),
            min="min",
            max="max",
        )

    return close_describe, summary, close_by_year


### 3.3. Evaluación de resultados

In [14]:
close_desc, close_summary, close_yearly = analyze_close_statistics(mnq_intraday)

print(close_desc)
#print(close_summary)
#print(close_yearly)


count    744013.000000
mean      14762.034275
std        3566.973264
min        6765.750000
1%         8099.780000
5%         9110.500000
25%       12065.250000
50%       14430.750000
75%       17513.000000
95%       21284.000000
99%       21908.720000
max       22317.250000
Name: close, dtype: float64


Algunas conclusiones del análisis de la variable `close`:


1. Nivel típico del índice MNQ

    La media de la variable `close` se ubica en 14 762 puntos, con una mediana cercana (14 431 puntos).
    Esto indica que, a lo largo de todo el período analizado, el MNQ ha operado la mayor parte del tiempo en torno al nivel de 15 000 puntos, validando dicho valor como referencia central de escala.

2. Amplio rango de valores y múltiples regímenes

    El rango observado va desde 6 766 hasta 22 317 puntos, lo que evidencia:

    - La presencia de múltiples regímenes de mercado
    - Una evolución estructural del índice a lo largo del tiempo
    - La no estacionariedad del nivel de precios.

    Este comportamiento descarta el uso de un único valor fijo de referencia para todo el período.

3. Concentración de observaciones en un rango operativo claro

    El 50 % central de los datos (P25–P75) se encuentra entre 12 065 y 17 513 puntos, mientras que el 90 % de las observaciones (P5–P95) se concentra aproximadamente entre 9 110 y 21 284 puntos.

    Esto define un rango operativo predominante, dentro del cual se desarrollan la mayoría de las oportunidades intradía.

4. Implicancia directa sobre la conversión puntos ↔ retornos

    Dado que el mismo desplazamiento en puntos genera retornos distintos según el nivel de `close`, la variabilidad observada implica que:

    - Un objetivo fijo en puntos (ej. 25 o 60 puntos) no corresponde a un retorno fijo.
    - Los umbrales de retorno deben ser dinámicos y dependientes de close_t.

    Esta conclusión es central para evitar sesgos de escala en la definición posterior de targets.

5. Consistencia con el objetivo económico del proyecto

    El rango y la media observados son coherentes con los objetivos definidos en el Punto 1, ya que:

    - Los niveles típicos del índice permiten que movimientos de 25 a 60 puntos representen retornos intradía realistas.

    -Dichos movimientos no corresponden a eventos extremos en la mayor parte del período analizado.

**Conclusión general**

El análisis de la variable close confirma que:

- El nivel del índice MNQ presenta una variabilidad significativa pero acotada,
- La media cercana a 15 000 puntos es representativa a nivel global,
- Cualquier definición de retornos objetivo debe considerar close como factor de escala dinámico.

Con estas conclusiones establecidas, el siguiente paso lógico es evaluar cómo se comportan los retornos logarítmicos en relación con estos niveles de precio, y con qué frecuencia permiten alcanzar los objetivos económicos definidos.

## **4. Evaluación de los retornos logarítmicos**


Una vez establecidos los parámetros económicos de la operatoria (Punto 2) y analizado el nivel y la variabilidad del índice MNQ a través de la variable `close` (Punto 3), corresponde evaluar el comportamiento estadístico de los retornos logarítmicos, con el objetivo de determinar si los movimientos necesarios para alcanzar los objetivos económicos definidos ocurren con una frecuencia razonable.

Para este análisis se consideran los retornos acumulados a distintos horizontes temporales (`ret_60`, `ret_90`), los cuales representan la variación relativa del precio del índice en ventanas de 60 y 90 minutos, respectivamente.

### **4.1. Justificación del uso de retornos**

El análisis del movimiento del índice MNQ no se realiza directamente sobre el precio (`close`), sino sobre sus retornos logarítmicos, debido a razones metodológicas y financieras bien establecidas.

En particular, los retornos logarítmicos:

- permiten comparar movimientos de precios en distintos niveles del índice,
- son aditivos en el tiempo, lo que facilita el análisis en ventanas temporales,
- presentan propiedades estadísticas más estables que los precios absolutos,
- constituyen la forma estándar de modelar variaciones relativas en finanzas cuantitativas.

Dado que el objetivo del proyecto es evaluar movimientos relativos del mercado en horizontes intradía, el retorno logarítmico resulta una representación más adecuada que la variación absoluta del precio.

### **4.2 Definición y cálculo de los retornos**

El retorno logarítmico se define como:

$$
r_t = \ln\left( \frac{close_{t+h}}{close_t} \right)
$$

donde:

- `close_t` es el valor del índice en el instante actual,
- `close_{t+h}` es el valor del índice luego de un horizonte temporal
`h`.


En este proyecto se consideran retornos acumulados en cuatro horizontes:

- `ret_60`: retorno a 60 minutos
- `ret_90`: retorno a 90 minutos


Estos retornos representan la variación relativa del índice en ventanas temporales alineadas con la operatoria intradía planteada en el Punto 1.

#### **3.2.1. Cálculo de retornos por horizonte temporal**

In [15]:
def add_log_return(df):
    df['ret_60'] = df.groupby('date')['close'].transform(
        lambda x: np.log(x.shift(-60)) - np.log(x)
    )

    df['ret_90'] = df.groupby('date')['close'].transform(
        lambda x: np.log(x.shift(-90)) - np.log(x)
    )
    return df

In [16]:
mnq_intraday_with_returns = mnq_intraday.copy()
mnq_intraday_with_returns = add_log_return(mnq_intraday_with_returns)

In [17]:
mnq_intraday_with_returns.head()

,date,open,high,low,close,volume,ret_60,ret_90
datetime,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,0.001031,0.000687
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,0.001059,0.000859
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,0.001117,0.000916
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,0.000974,0.000916
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,0.000916,0.000888


### **4.3. Análisis descriptivo de retornos logarítmicos por horizonte temporal**

#### **4.3.1. Código de aplicación**

In [18]:
returns = ['ret_60', 'ret_90']

In [19]:
returns_stats = (
    mnq_intraday_with_returns[returns]
    .dropna()
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)

#### **4.3.2. Análisis y Conclusiones**

In [20]:
returns_stats

,ret_60,ret_90
count,626743.000000,626743.000000
mean,0.000051,0.000076
std,0.004264,0.005267
min,-0.056549,-0.049889
1%,-0.012411,-0.015035
5%,-0.006740,-0.008441
50%,0.000189,0.000244
95%,0.006147,0.007738
99%,0.011407,0.013842
max,0.079896,0.083184


1. Coherencia estadística entre horizontes

    Se observa un comportamiento consistente y esperable al ampliar el horizonte temporal:
    - La media del retorno aumenta levemente al pasar de 60 a 90 minutos.
    - El desvío estándar crece de forma monótona (60 → 90 min).

    Esto confirma que:
    - los retornos están correctamente calculados,
    - horizontes más largos capturan una mayor acumulación de movimiento.

2. Media cercana a cero (propiedad intradía)

    En ambos horizontes, la media del retorno es muy próxima a cero, lo que indica que:
    - no existe un sesgo direccional sistemático,
    - el mercado intradía es esencialmente balanceado en promedio.

    Conclusión metodológica:
    - no es apropiado modelar retornos esperando una tendencia promedio,
    - el valor predictivo reside en eventos específicos y contextuales, no en el promedio global.

    Una media cercana a cero no implica ausencia de estructura, sino ausencia de una tendencia promedio explotable sin condicionamiento.

3. Distribuciones balanceadas con ligera asimetría positiva

    Los percentiles negativos y positivos presentan magnitudes comparables, con:
    - colas negativas y positivas de tamaño similar,
    - una **ligera asimetría positiva**, consistente con la naturaleza alcista de los índices bursátiles.

    Esto habilita:
    - modelos simétricos long/short,
    - formulaciones de clasificación direccional sin sesgo estructural fuerte.

4. Incremento de magnitud con el horizonte temporal

    Los percentiles altos crecen de forma clara al ampliar el horizonte:

    - Percentiles 95:
      - ret_60 ≈ 0.62 %
      - ret_90 ≈ 0.78 %

    - Percentiles 99:
      - ret_60 ≈ 1.14 %
      - ret_90 ≈ 1.38 %

    Conclusión:
    - los movimientos económicamente relevantes existen,
    - su frecuencia y magnitud dependen fuertemente del horizonte temporal considerado.

5. Presencia de colas extremas

    Los valores mínimos y máximos muestran:
    - retornos extremos del orden de ±5 % a ±9 %,
    - asociados a eventos excepcionales (aperturas, noticias, shocks de mercado).

    Implicación:
    - estos eventos no deben tomarse como referencia operativa base,
    - pero confirman que el dataset captura escenarios de estrés reales y completos.

6. Interpretación económica implícita (MNQ)

    Dado que el contrato MNQ equivale a 2 USD por punto, los percentiles altos implican:
    - movimientos típicos (P95) económicamente operables,
    - escenarios de alta expansión (P99) coherentes con targets superiores a 90 puntos.

    Esto valida la consistencia entre:
    - el análisis de retornos,
    - y la posterior definición de objetivos económicos en puntos (delta_pts).

7. Rol de cada horizonte en el pipeline

    A partir del análisis estadístico (sin definir aún targets operativos):
    - 60 minutos: buen equilibrio entre frecuencia y amplitud del movimiento.
    - 90 minutos: movimientos de mayor magnitud, menor frecuencia y escenarios de continuidad.

    Esto permite, en el siguiente paso, alinear objetivos económicos y reglas operativas con cada horizonte, sin forzar supuestos ni introducir sesgos artificiales.

8. Implicación metodológica para el modelado

    Si bien los retornos son adecuados para análisis estadístico global, la operativa intradía se beneficia de trabajar en puntos absolutos, dado que:
    - los stops y targets son lineales,
    - el riesgo monetario es explícito,
    - y la interpretación es directa para ejecución real.

    Por este motivo, el pipeline evoluciona desde retornos (ret_h) hacia movimientos en puntos (delta_pts_h) como variable central de decisión.


**Conclusión general**

El análisis descriptivo de los retornos logarítmicos muestra que:

- el mercado del MNQ presenta movimientos intradía de magnitud creciente a medida que se amplía el horizonte temporal,
- los retornos exhiben propiedades estadísticas sanas y coherentes,
- existen movimientos suficientemente amplios para sustentar objetivos económicos razonables,
- aunque dichos movimientos no se presentan de forma uniforme ni constante en el tiempo.

Desde el punto de vista metodológico, este análisis valida el uso de los retornos como herramienta exploratoria y estadística, pero no como variable operativa directa.

En consecuencia, el siguiente paso lógico del pipeline es traducir estas magnitudes de retorno a movimientos absolutos en puntos, que permiten definir de manera explícita targets, stops y relaciones riesgo–beneficio, y recién entonces establecer los objetivos operativos finales.

## **5. Vinculación entre nivel de precio (`close`), retornos y objetivos económicos**

### **5.1. Explicación metodológica**

El objetivo de este punto es integrar los resultados obtenidos en los Puntos 3 y 4 con los parámetros económicos definidos en el Punto 2, de modo de establecer una relación cuantitativa coherente entre:

- el nivel del índice MNQ (`close`),
- los retornos logarítmicos (`ret_**`) intradía,
- y los objetivos económicos expresados en puntos y dólares.

Esta vinculación es un paso intermedio indispensable antes de definir formalmente los targets de predicción.

**1. Principio de correspondencia entre puntos y retornos**

  Los objetivos económicos del proyecto se formulan en términos de variación absoluta del índice (puntos), mientras que el comportamiento estadístico del mercado se analiza mediante retornos logarítmicos. Para conectar ambos dominios se utiliza la relación de correspondencia:

  $$
    \Delta \text{puntos} \approx r_{t,h} \times close_t
  $$
      
  donde:

  - `𝑟_{𝑡,ℎ}` es el retorno logarítmico acumulado en un horizonte `ℎ`
  - `𝑐𝑙𝑜𝑠𝑒_𝑡` es el nivel del índice al inicio de la ventana.

  Esta expresión permite traducir cualquier retorno observado a una variación en puntos directamente interpretable desde el punto de vista operativo.
<br><br>
**2. Uso del nivel de precio como factor de escala dinámico**

Dado que el análisis de la variable `close` evidenció una variabilidad significativa del nivel del índice a lo largo del tiempo, la conversión entre retornos y puntos no puede basarse en un valor fijo de referencia.

Por el contrario:
- cada observación debe evaluarse en función de su propio `close_t`,
- los umbrales de retorno asociados a un objetivo en puntos se definen de forma dinámica y dependiente del nivel del mercado.

Este enfoque garantiza que:
- los criterios operativos sean coherentes en distintos regímenes de precio,
- los resultados no estén sesgados hacia períodos específicos del dataset.
<br><br>
**3. Integración de horizontes temporales y objetivos operativos**

La vinculación entre precio y retorno se realiza de manera conjunta con el horizonte temporal del movimiento, dado que:

- distintos horizontes presentan distintas combinaciones de frecuencia y magnitud,
- los objetivos económicos definidos requieren movimientos de cierta amplitud en ventanas temporales compatibles con la operatoria intradía.

En consecuencia, esta etapa no busca identificar un único horizonte óptimo, sino evaluar cómo cada horizonte contribuye al cumplimiento de los objetivos económicos, considerando tanto la magnitud del movimiento como su frecuencia histórica.
<br><br>
**4. Rol de esta vinculación en la definición de targets**

La finalidad de este punto no es aún fijar los targets, sino:

- determinar qué rangos de retorno son compatibles con los objetivos económicos definidos,
- identificar qué horizontes temporales concentran dichos movimientos,
- establecer una base cuantitativa objetiva para la definición posterior de los targets de predicción.
<br><br>
**Cierre del punto (explicativo)**

En síntesis, la vinculación entre el nivel de precio, los retornos logarítmicos y los objetivos económicos permite traducir los requerimientos operativos del proyecto al lenguaje estadístico del dataset, asegurando coherencia entre:

- la formulación del problema,
- el comportamiento histórico del mercado,
- y la realidad de la operatoria intradía.

### **5.2. Análisis cuantitativo top-down: desde los targets empíricos hacia los retornos implícitos**




Una vez definidos empíricamente los targets operativos en la notebook 03a, el objetivo de este subpunto es validar cuantitativamente dichos valores desde una perspectiva top-down, evaluando:

- qué retornos logarítmicos implícitos requieren,
- cómo dependen del nivel del índice (close),
- y en qué región de la distribución histórica de retornos se ubican.

Este análisis no busca redefinir targets, sino verificar su coherencia estadística y económica.
<br>

**1. Conversión de targets empíricos a retornos implícitos**

Dado un target empírico `Δ` (en puntos) y un horizonte `ℎ`, el retorno logarítmico implícito requerido puede expresarse como:

$$
r_{\text{Δ,t,h}} \approx \ln\left(1 + \frac{\Delta}{close_t}\right) \approx \frac{\Delta}{close_t}
$$

En el contexto intradía del MNQ, donde los movimientos relativos son pequeños, se utiliza la aproximación:

$$
r_{\text{Δ,t,h}} \approx \frac{\Delta}{close_t}
$$


Los targets empíricos considerados son:

- Horizonte 60 min: $Δ_{60} \approx 84.14$ pts.
- Horizonte 90 min: $Δ_{90} \approx 97.22$ pts.
<br>

**2. Umbrales de retorno equivalentes usando percentiles de `close`**

Dado que el nivel del índice MNQ varía significativamente a lo largo del período analizado, la conversión puntos → retorno se evalúa en distintos regímenes representativos de `close`:

  - P05 close = 9 110.5
  - P50 close = 14 430.75
  - P95 close = 21 284.0

**Horizonte 60 min — Target ≈ 84.18 pts**

$$r_{60} \approx \frac{84.14}{close_t}$$

En P05: $$\frac{84.14}{9110.5} = 0.00924 \;\Rightarrow\; 0.924\%$$
En P50: $$\frac{84.14}{14430.75} = 0.00583 \;\Rightarrow\; 0.583\%$$
En P95: $$\frac{84.14}{21284} = 0.00395 \;\Rightarrow\; 0.395\%$$
<br>

**Horizonte 90 min — Target ≈ 97.22 pts**

$$r_{90} \approx \frac{97.22}{close_t}$$

En P05: $$\frac{97.22}{9110.5} = 0.01067 \;\Rightarrow\; 1.627\%$$
En P50: $$\frac{97.22}{14430.75} = 0.00673 \;\Rightarrow\; 0.673\%$$
En P95: $$\frac{97.22}{21284} = 0.00457 \;\Rightarrow\; 0.457\%$$
<br>

Estos valores representan los retornos implícitos típicos necesarios para alcanzar los targets empíricos bajo distintos niveles de precio del mercado.
<br>

**3. Ubicación de los targets empíricos en la distribución de retornos**

Los percentiles positivos de la distribución histórica de retornos muestran:

- `ret_60`:
  - P95 ≈ 0.615 %
  - P99 ≈ 1.135 %

- `ret_90`:
  - P95 = 0.775%
  - P99 = 1.378%

La comparación directa indica que:

- el target de 84 pts (H60) se ubica:
  - cerca del P95 en regímenes medios,
  - por debajo del P95 en regímenes altos,
  - claramente por debajo del P99 en todos los casos.

- el target de 97 pts (H90):
  - se sitúa entre P95 y P99 en regímenes bajos,
  - alrededor de P95 en regímenes medios,
  - por debajo de P95 en regímenes altos.

**4. Conclusión del análisis top-down**

El análisis cuantitativo muestra que los targets empíricos definidos en la notebook 03a:
- requieren retornos significativos pero no extremos,
- se ubican en regiones de la distribución históricamente observables,
- y son coherentes con la frecuencia y magnitud de los movimientos intradía del MNQ.

En consecuencia, los targets seleccionados no son arbitrarios, sino que se encuentran **bien posicionados dentro de la estructura estadística del mercado**, validando su uso como objetivos operativos y como base para la definición final de targets de predicción.

### **5.3. Evaluación de la frecuencia empírica de movimientos económicamente relevantes (desde el objetivo hacia la frecuencia real)**





#### **5.3.1. Marco teórico**

La pregunta a responder ahora es: **¿es viable el objetivo planteado?**

Hasta este punto del análisis se ha establecido:

- Cuántos puntos por operación son necesarios para cumplir los objetivos operativos (Punto 1).
- En qué rangos de precio opera el índice a lo largo del tiempo (Punto 2).
- Cómo se distribuyen los retornos intradía para distintos horizontes temporales (Punto 3).

Sin embargo, aún resta responder una cuestión central: con qué frecuencia real el mercado alcanza dichos objetivos.

Este sub-punto tiene como finalidad responder a la siguiente pregunta clave:

**¿Con qué probabilidad histórica el MNQ se mueve al menos Δ puntos dentro de un horizonte intradía dado?**

---

**La frecuencia como criterio decisivo**

La existencia de un movimiento no implica necesariamente su viabilidad operativa. En particular, un desplazamiento del precio puede:

- Existir desde el punto de vista estadístico.
- Presentar una magnitud compatible con los objetivos planteados.
- Pero ocurrir con una frecuencia demasiado baja para sostener una operatoria diaria.

Por lo tanto:
- No es suficiente saber que un evento puede ocurrir.
- Es imprescindible conocer cada cuántas veces ocurre.

La frecuencia empírica es el factor que permite determinar:
- Si el objetivo mínimo es alcanzable de forma consistente.
- Si el objetivo ideal es realista o meramente excepcional.
- Cuántas operaciones diarias tienen sentido desde una perspectiva probabilística.

----

**Uso de un umbral dinámico de evaluación**

Dado que:
- Los objetivos operativos se definen en puntos absolutos (Δ).
- El índice opera en niveles de precio variables a lo largo del tiempo.

No resulta adecuado evaluar la condición mediante un umbral fijo del tipo:
`ret_h ≥ constante`

En su lugar, se utiliza un umbral dinámico, definido como:

$$ret_{t,h} \ge \frac{\Delta}{close_t}$$

donde:

- `Δ` definidos empíricamente en 03a (p. ej. 84.18 para H60 y 97.22 para H90).
- `close_t` es el nivel del índice en el instante inicial `𝑡`.

Este enfoque garantiza que:
- La comparación sea homogénea en distintos regímenes de precio.
- No se sobreestimen ni subestimen oportunidades en función del nivel del índice.
- El análisis permanezca alineado con la lógica de la operatoria real.

---

Este análisis permite:

1. Cuantificar la viabilidad real de cada objetivo en cada horizonte.
2. Identificar qué horizontes temporales concentran más oportunidades.
3. Separar:
    - objetivos frecuentes y operables,
    - de objetivos eventuales o de extensión.
4. Fundamentar la futura definición de targets con evidencia empírica, no con supuestos.

En síntesis, este sub-punto transforma el análisis previo en una medida clave para la toma de decisiones: la frecuencia histórica de movimientos económicamente relevantes. Solo a partir de esta información es posible definir targets que sean estadísticamente defendibles y operativamente sostenibles.

#### **5.3.2. Cálculo de frecuencias**

Para cada horizonte $h \in \{60, 90\}$ y para cada objetivo expresado en puntos $\Delta$, se evalúa las siguientes condiciones:

- Movimiento sin dirección (recomendado para viabilidad):

$$
|ret_{t,h}| \ge \frac{\Delta}{close_t}
$$

- Direccional (si quiere long/short separado):

$$
ret_{t,h} \ge \frac{\Delta}{close_t}  (LONG)
$$

$$
ret_{t,h} \le -\frac{\Delta}{close_t}  (SHORT)
$$

La frecuencia empírica se define como el porcentaje de observaciones que cumplen dicha condición respecto del total de observaciones válidas.

Este análisis responde directamente a la pregunta:

**“¿En qué proporción de los casos el mercado se movió al menos $\Delta$ puntos en $h$ minutos?”**

In [21]:
import numpy as np
import pandas as pd

def compute_move_frequencies(
    df: pd.DataFrame,
    close_col: str = "close",
    return_cols=("ret_60", "ret_90"),
    point_targets=(84.18, 97.22),
    exact_log: bool = False
) -> pd.DataFrame:
    """
    Frecuencias empíricas de alcanzar un objetivo Δ (en puntos) en horizonte h.

    Mide:
      - freq_long : P(ret_h >=  thr)
      - freq_short: P(ret_h <= -thr)
      - freq_move : P(|ret_h| >= thr)  (movimiento sin dirección)

    Umbral dinámico por observación:
      thr ≈ Δ / close_t           (aprox intradía)
      thr = ln(1 + Δ/close_t)     (exacto log) si exact_log=True
    """

    results = []

    for ret_col in return_cols:
        sub = df[[close_col, ret_col]].dropna()
        if sub.empty:
            continue

        close_t = sub[close_col].astype(float).values
        ret_t = sub[ret_col].astype(float).values

        for pts in point_targets:
            if exact_log:
                thr = np.log1p(pts / close_t)          # ln(1 + Δ/close)
            else:
                thr = (pts / close_t)                  # aprox Δ/close

            freq_long  = np.mean(ret_t >=  thr)
            freq_short = np.mean(ret_t <= -thr)
            freq_move  = np.mean(np.abs(ret_t) >= thr)

            results.append({
                "ret_col": ret_col,
                "h_minutes": int(ret_col.split("_")[1]),
                "points_target": float(pts),
                "threshold_mode": "exact_log" if exact_log else "approx",
                "freq_long": float(freq_long),
                "freq_short": float(freq_short),
                "freq_move_abs": float(freq_move),
            })

    out = pd.DataFrame(results).sort_values(["h_minutes", "points_target"])
    return out


In [22]:
freq_delta_60 = compute_move_frequencies(
    mnq_intraday_with_returns,
    point_targets=(delta_base_60, delta_op_60)
)

NameError: name 'delta_base_60' is not defined

In [ ]:
freq_delta_90 = compute_move_frequencies(
    mnq_intraday_with_returns,
    point_targets=(delta_base_90, delta_op_90)
)

#### **5.3.3. Resultados y análisis**

##### **Deltas de H=60**

In [ ]:
print('Para los deltas de H=60 - Calculado para retornos de 60 y 90min\n')

print(f'\tdelta_base_60: {delta_base_60}')
print(f'\tdelta_op_60: {delta_op_60}\n')
freq_delta_60

**Para `ret_60` (h = 60 min)**

Δ = 52.12 pts (delta base)
- LONG: 12.66 %
- SHORT: 12.85 %
- MOVE ( |ret| ≥ Δ ): 25.52 %
  
  → aproximadamente 1 de cada 4 ventanas de 60 minutos

Δ = 84.18 pts (delta operativo / p70)
- LONG: 5.32 %
- SHORT: 6.32 %
- MOVE ( |ret| ≥ Δ ): 11.64 %
  
  → aproximadamente 1 de cada 8.6 ventanas de 60 minutos

Interpretación:

El target operativo es claramente menos frecuente que el delta base, pero mantiene una frecuencia de doble dígito, lo que lo hace operativamente viable.

**Para `ret_90` (h = 90 min), usando los mismos Δ**

Δ = 52.12 pts
- LONG: 17.42 %
- SHORT: 16.87 %
- MOVE ( |ret| ≥ Δ ): 34.29 %

  → aproximadamente 1 de cada 2.9 ventanas de 90 minutos

Δ = 84.18 pts
- LONG: 8.51 %
- SHORT: 9.48 %
- MOVE ( |ret| ≥ Δ ): 17.99 %
  
  → aproximadamente 1 de cada 5.6 ventanas de 90 minutos

Interpretación:

Al aumentar el horizonte temporal, crece la frecuencia empírica de alcanzar un mismo delta, confirmando la relación esperada entre horizonte y probabilidad de movimiento.

##### **Deltas de H=90**

In [ ]:
print('Para los deltas de H=90 - Calculado para retornos de 60 y 90min\n')

print(f'\tdelta_base_90: {delta_base_90}')
print(f'\tdelta_op_90: {delta_op_90}\n')
freq_delta_90

**Para `ret_60` (h = 60 min)**

Δ = 60.75 pts (delta base H90)
- LONG: 9.92 %
- SHORT: 10.59 %
- MOVE ( |ret| ≥ Δ ): 20.51 %
  
  → aproximadamente 1 de cada 4.9 ventanas de 60 minutos

Δ = 97.22 pts (delta operativo H90 / p70)
- LONG: 3.81 %
- SHORT: 4.79 %
- MOVE ( |ret| ≥ Δ ): 8.60 %
  
  → aproximadamente 1 de cada 11.6 ventanas de 60 minutos

  Interpretación:
  Cuando se exige un target propio del horizonte 90 sobre ventanas de 60 minutos, la frecuencia cae de forma significativa, lo que indica una incompatibilidad natural entre magnitud objetivo y horizonte corto.

**Para `ret_90` (h = 90 min)**

Δ = 60.75 pts (delta base H90)
- LONG: 14.24 %
- SHORT: 14.47 %
- MOVE ( |ret| ≥ Δ ): 28.71 %
  
  → aproximadamente 1 de cada 3.5 ventanas de 90 minutos

Δ = 97.22 pts (delta operativo H90 / p70)
- LONG: 6.35 %
- SHORT: 7.47 %
- MOVE ( |ret| ≥ Δ ): 13.82 %
  → aproximadamente 1 de cada 7.2 ventanas de 90 minutos

  Interpretación:
  Cuando el horizonte temporal es consistente con el delta definido, el target operativo mantiene una frecuencia de doble dígito, confirmando su viabilidad operativa dentro de la lógica del horizonte de 90 minutos.

#### **5.3.4. Comparación H=60 y H=90**

A partir de los resultados obtenidos en los subpuntos anteriores, se realiza una comparación directa entre los horizontes H = 60 minutos y H = 90 minutos, considerando como criterios principales:

- Magnitud del target operativo (Δ en puntos),
- Frecuencia empírica de ocurrencia,
- Consistencia entre horizonte y objetivo económico.

**Horizonte H = 60 minutos**

- Delta operativo: Δ ≈ 84.18 pts
- Frecuencia empírica (ret_60):
  - MOVE (|ret| ≥ Δ): ≈ 11.6 %

    → ~1 de cada 8.6 ventanas

- Frecuencia empírica (ret_90):
  - MOVE (|ret| ≥ Δ): ≈ 18.0 %

    → ~1 de cada 5.6 ventanas

Lectura:

- El target operativo presenta una frecuencia moderada, pero estable.
- El horizonte de 60 minutos exige un timing más preciso, ya que pequeñas desviaciones temporales reducen sensiblemente la probabilidad de éxito.
- La ventana óptima identificada es más estrecha, lo que incrementa la dependencia temporal.

**Horizonte H = 90 minutos**

- Delta operativo: Δ ≈ 97.22 pts
- Frecuencia empírica (ret_90):
  - MOVE (|ret| ≥ Δ): ≈ 13.8 %
    
    → ~1 de cada 7.2 ventanas

- Frecuencia empírica (ret_60):
  - MOVE (|ret| ≥ Δ): ≈ 8.6 %

    → ~1 de cada 11.6 ventanas

Lectura:

- A pesar de requerir una mayor magnitud absoluta, el horizonte de 90 minutos mantiene una frecuencia empírica comparable o superior a H=60.
- La ventana temporal asociada es más amplia y estable, reduciendo la sensibilidad al punto exacto de entrada.
- Existe una mejor alineación natural entre magnitud objetivo y duración del horizonte

**Análisis sintético**

| Criterio                     | H = 60 min | H = 90 min |
|-----------------------------|------------|------------|
| Delta operativo              | ~84 pts    | ~97 pts    |
| Frecuencia MOVE (h consistente) | ~11.6 %   | ~13.8 %   |
| Dependencia temporal         | Alta       | Moderada   |
| Tolerancia al timing         | Baja       | Alta       |
| Estabilidad operativa        | Media      | Alta       |

Aunque ambos horizontes son operables, el análisis conjunto de frecuencia, magnitud y estabilidad temporal indica que:

- **H = 90 minutos ofrece un mejor equilibrio estructural:**
    - targets de mayor magnitud,
    - frecuencia empírica suficiente,
    - y menor dependencia del timing exacto.

En consecuencia, el horizonte de 90 minutos se adopta como horizonte operativo principal, manteniendo H = 60 minutos como referencia secundaria y comparativa dentro del pipeline.


### **5.4. Conclusión general del Punto 5**

El desarrollo de este punto permitió integrar de manera coherente:

- el comportamiento estadístico de los retornos intradía,
- la dependencia del nivel de precio del índice MNQ,
- y los objetivos económicos definidos en términos de puntos y dólares.

A través de los análisis top-down y de frecuencia empírica, se validó que los targets operativos adoptados se ubican en regiones de la distribución históricamente observables y presentan una frecuencia compatible con una operatoria intradía sostenible.

El análisis empírico inverso de los movimientos intradía en puntos fue realizado exhaustivamente en la notebook stage_03a_target_investigation. En esta etapa, dichos resultados se adoptan como base definitiva para la definición de targets, evitando la duplicación de análisis y asegurando la coherencia metodológica del pipeline.

De este modo, el Punto 5 cumple su función de validación cuantitativa y enlace metodológico entre la investigación empírica previa y la implementación final de los targets de predicción en la notebook 03b.

## **6. Definición final de targets y parámetros operativos**

### **6.1. Marco conceptual**

**1. Principio general de definición del target de predicción**

La definición del target de predicción se fundamenta en la integración de tres criterios previamente establecidos:

1. **Criterio económico**: el modelo debe capturar movimientos intradía con relevancia económica real.  
2. **Criterio estadístico–probabilístico**: los movimientos modelados deben ocurrir con frecuencia suficiente para sostener una operatoria sistemática.  
3. **Criterio empírico de mercado**: los movimientos objetivo deben alinearse con las magnitudes que el mercado genera de forma natural, evitando tanto el ruido intradía como las colas extremas.

Bajo este marco, **el objetivo del modelo no es predecir un retorno puntual ni un umbral específico**, sino **modelar la dinámica futura del precio dentro de un horizonte temporal fijo**, a partir de la cual se derivan posteriormente decisiones operativas.

<br>

**2. Selección del horizonte temporal de predicción**

A partir del análisis empírico conjunto realizado, se concluye que:

- El horizonte de **90 minutos** ofrece el mejor compromiso entre:
  - frecuencia empírica,
  - magnitud del movimiento,
  - compatibilidad con una operatoria intradía.

En consecuencia, el **horizonte principal de predicción es H = 90 minutos**, dejando el horizonte de **60 minutos** como complementario o experimental.

<br>

**3. Definición del target primario del modelo (enfoque seq2seq)**

El target principal del modelo se define como la **secuencia completa de desplazamientos futuros del precio** dentro del horizonte considerado:

$$
\mathbf{y}_t = \{\Delta pts_{t+1}, \Delta pts_{t+2}, \dots, \Delta pts_{t+H}\}
$$

donde:

$$
H \in \{60, 90\}
$$

Este enfoque **seq2seq** permite:

- capturar la **estructura temporal** del movimiento,
- preservar información de **dirección, magnitud y progresión**,
- evitar la pérdida de información asociada a la binarización temprana del target.

El modelo aprende así una representación rica del comportamiento intradía futuro del MNQ.

<br>

**4. Definición de umbrales operativos derivados (post-predicción)**

A partir del análisis empírico realizado en la notebook *03a*, se identifican tres niveles característicos de desplazamiento acumulado dentro del horizonte:

**Movimiento estructural (base):**
- H = 60 min: Δ ≈ 52.12 pts (`delta_base_60`)
- H = 90 min: Δ ≈ 60.75 pts (`delta_base_90`)

**Movimiento operativo (p70):**
- H = 60 min: Δ ≈ 84.14 pts (`delta_op_60`)
- H = 90 min: Δ ≈ 97.22 pts (`delta_op_90`)

**Movimiento de extensión (p90):**
- H = 60 min: Δ ≈ 140.7 pts (`delta_tail_60`)
- H = 90 min: Δ ≈ 162.3 pts (`delta_tail_90`)

Estos valores:

- **no constituyen el target de entrenamiento del modelo**,  
- sino **umbrales operativos aplicados sobre la secuencia predicha**.

<br>

**5. Rol de los umbrales en la lógica operativa**

Dada una secuencia predicha $\hat{\mathbf{y}}_t$, se evalúan métricas agregadas (por ejemplo, desplazamiento acumulado, máximo excursionado o perfil temporal) para determinar si:

$$
\sum_{i=1}^{H} \hat{\Delta pts}_{t+i} \ge \Delta_{base},\ \Delta_{op},\ \Delta_{tail}
$$

De este modo:

- `delta_base_h` define el **movimiento mínimo estructural relevante**,  
- `delta_op_h` define el **objetivo principal de monetización**,  
- `delta_tail_h` actúa como **referencia de escenarios excepcionales**.

<br>

**6. Reformulación del problema de predicción**

Con base en lo anterior, **el problema se formula primariamente como un problema de regresión secuencial (seq2seq)**.

Las formulaciones de clasificación o estimación probabilística de umbrales:

$$
P(\Delta pts_{t,h} \ge \Delta)
$$

son **derivadas posteriores**, utilizadas exclusivamente para la toma de decisiones operativas y la evaluación del modelo, pero **no como objetivo directo de entrenamiento**.

<br>

**7. Rol del target dentro del pipeline**

El target seq2seq definido en este punto:

- constituye la **variable objetivo primaria del dataset**,  
- se utiliza en las etapas de:
  - entrenamiento,
  - validación,
  - comparación de arquitecturas,
- permite adaptar la lógica operativa a distintos regímenes de mercado **sin redefinir el target del modelo**.

<br>

**Cierre del marco conceptual**

La definición del target como una **secuencia futura de retornos en puntos**, junto con la derivación posterior de umbrales operativos empíricamente validados, permite desacoplar el aprendizaje estadístico del modelo de la lógica de ejecución.  
Este enfoque maximiza la información aprendida, reduce el sobreajuste y mantiene una alineación estricta con la dinámica real del MNQ y con una operatoria intradía sostenible.


### **6.2. Definición formal de etiquetas (labels) e implementación**

#### **6.2.1. Definición formal de etiquetas**

Para cada instante $t$ (cada fila intradía) y un horizonte temporal $h \in \{60, 90\}$ minutos, se define el **desplazamiento futuro del precio en puntos** como:

$$
\Delta pts_{t,h} = close_{t+h} - close_t
$$

Esta magnitud constituye el **objetivo de predicción del modelo**, siendo:

- **$h = 90$ minutos** el **horizonte principal de modelado**,  
- **$h = 60$ minutos** un **horizonte complementario o experimental**.

A partir de la secuencia de valores $\Delta pts_{t,h}$ predicha por el modelo, se derivan posteriormente **etiquetas y criterios direccionales u operativos**, definidos mediante umbrales económicamente relevantes (`delta_base_h`, `delta_op_h`, `delta_tail_h`), los cuales han sido estimados empíricamente en la notebook *03a* y **no forman parte del target directo de entrenamiento**.


**Umbrales utilizados**

- Umbral base (entrada): $Δ_{𝑏𝑎𝑠𝑒,ℎ} ∈ {[52.12, 60.75]} $
- Umbral operativo (objetivo): $Δ_{op,ℎ} ∈ {[84.18, 97.22]} $
- Umbral excepcional (extensivo): $Δ_{tail,ℎ} ∈ {[140.7, 162.3]} $

**A) Etiqueta de entrada direccional (trade_h)**

La etiqueta `trade_h` indica si corresponde abrir una operación y su dirección, indica si existe una condición mínima de movimiento estructural y su dirección:

$$
\text{trade}_h =
\begin{cases}
1 & \text{si } \Delta pts_{t,h} \ge \Delta_{\text{base,h}} (BUY)\\
-1 & \text{si } \Delta pts_{t,h} \le -\Delta_{\text{base,h}} (SELL) \\
0 & \text{si }  -\Delta_{\text{base}} < |\Delta pts_{t,h}| < \Delta_{\text{base,h}} (FLAT) \\
\end{cases}
$$

`trade_h` no garantiza ganancia, sino que identifica escenarios con potencial operativo mínimo.

Esta etiqueta constituye el target principal del modelo de entrada.


**B) Etiqueta de objetivo operativo (hit_op_h)**

La etiqueta `hit_op_h` indica si, dado que existe una señal válida, el movimiento alcanza el objetivo operativo de monetización:

$$
\text{hit_op}_h =
\begin{cases}
1 & \text{si } \text{trade}_h = +1 \;\text{y}\; \Delta pts_{t,h} \ge \Delta_{\text{op,h}} \\
1 & \text{si } \text{trade}_h = -1 \;\text{y}\; \Delta pts_{t,h} \le -\Delta_{\text{op,h}} \\
0 & \text{en caso contrario}
\end{cases}
$$

**C) Etiqueta de extensión (hit_tail_h)**

La etiqueta `hit_tail_h` indica si el movimiento alcanza una expansión excepcional:

$$
\text{hit_tail}_h =
\begin{cases}
1 & \text{si } \text{trade}_h = +1 \;\text{y}\; \Delta pts_{t,h} \ge \Delta_{\text{tail,h}} \\
1 & \text{si } \text{trade}_h = -1 \;\text{y}\; \Delta pts_{t,h} \le -\Delta_{\text{tail,h}} \\
0 & \text{en caso contrario}
\end{cases}
$$

---
**Restricción operativa**

Adicionalmente, para evitar inconsistencias operativas, imponemos la restricción:
$$
\text{trade}_h = 0 \;\Rightarrow\; \text{hit_op}_h = 0 \;\Rightarrow\; \text{hit_tail}_h = 0
$$

---
**Interpretación operativa**

Este esquema permite:

- Entrenar un modelo principal de entrada $$\text{trade}_h ∈ \text{{-1, 0 , 1}} $$
- Evaluar la probabilidad de monetización (`hit_op_h`).
- Y analizar la captura de colas (`hit_tail_h`).

Manteniendo una separación clara entre entrada, objetivo y extensión, sin asumir garantías implícitas de ganancia.



#### **6.2.2. Implementación en código para generar `mnq_intraday_labeled`**

El siguiente código hace lo siguiente:

1. Calcula Δ𝑝𝑡𝑠 a futuro por día (sin cruzar días)
2. Crea labels direccionales base y extensión para horizontes 60 y 90
3. Devuelve el dataset final labeled ya filtrado de NaNs de futuro.

In [ ]:
import numpy as np
import pandas as pd

def make_mnq_intraday_labeled(
    df: pd.DataFrame,
    date_col: str = "date",
    close_col: str = "close",
    horizons=(60, 90),
    delta_base_by_h=None,   # {60: 52.12, 90: 60.75}
    delta_op_by_h=None,     # {60: 84.18, 90: 97.22}
    delta_tail_by_h=None,   # {60: 140.7, 90: 162.3}
    drop_na_targets: bool = True,
    keep_delta: bool = True,
) -> pd.DataFrame:
    """
    Genera mnq_intraday_labeled con etiquetas basadas en deltas empíricos (03a).

    Para cada horizonte h:
      Δpts_{t,h} = close_{t+h} - close_t   (sin cruzar días)

      trade_h:
        +1 si Δpts_{t,h} >=  delta_base_h   (BUY)
        -1 si Δpts_{t,h} <= -delta_base_h   (SELL)
         0 en caso contrario                (FLAT)

      target_op_h:
        1 si trade_h=+1 y Δpts_{t,h} >=  delta_op_h
        1 si trade_h=-1 y Δpts_{t,h} <= -delta_op_h
        0 en caso contrario

      target_tail_h:
        1 si trade_h=+1 y Δpts_{t,h} >=  delta_tail_h
        1 si trade_h=-1 y Δpts_{t,h} <= -delta_tail_h
        0 en caso contrario

      Restricción operativa:
        trade_h=0 => target_op_h=0 y target_tail_h=0
    """
    out = df.copy()

    # Validaciones mínimas
    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' en el DataFrame.")
    if close_col not in out.columns:
        raise KeyError(f"Falta la columna '{close_col}' en el DataFrame.")

    # Defaults obligatorios (si no los pasan)
    if delta_base_by_h is None or delta_op_by_h is None or delta_tail_by_h is None:
        raise ValueError(
            "Debe proveer delta_base_by_h, delta_op_by_h y delta_tail_by_h "
            "como dicts, por ejemplo: {60: 52.12, 90: 60.75}."
        )

    # Orden por seguridad
    out = out.sort_index()

    for h in horizons:
        if h not in delta_base_by_h or h not in delta_op_by_h or h not in delta_tail_by_h:
            raise KeyError(f"Faltan deltas empíricos para h={h} en uno de los diccionarios.")

        base_h = float(delta_base_by_h[h])
        op_h = float(delta_op_by_h[h])
        tail_h = float(delta_tail_by_h[h])

        future_close = out.groupby(date_col, sort=False)[close_col].shift(-h)
        delta = future_close - out[close_col]

        if keep_delta:
            out[f"delta_pts_{h}"] = delta

        # trade_h (entrada)
        trade = np.select(
            [delta >= base_h, delta <= -base_h],
            [1, -1],
            default=0
        ).astype("int8")
        out[f"trade_{h}"] = trade

        # target_op_h (monetización)
        target_op = np.where(
            ((trade == 1) & (delta >= op_h)) |
            ((trade == -1) & (delta <= -op_h)),
            1, 0
        ).astype("int8")
        out[f"hit_op_{h}"] = target_op

        # target_tail_h (extensión / cola)
        target_tail = np.where(
            ((trade == 1) & (delta >= tail_h)) |
            ((trade == -1) & (delta <= -tail_h)),
            1, 0
        ).astype("int8")
        out[f"hit_tail_{h}"] = target_tail

        # Restricción operativa explícita (por robustez)
        flat_mask = (trade == 0)
        out.loc[flat_mask, f"target_op_{h}"] = 0
        out.loc[flat_mask, f"target_tail_{h}"] = 0

    # Drop de filas sin futuro (últimos h minutos de cada día)
    if drop_na_targets:
        if keep_delta:
            delta_cols = [f"delta_pts_{h}" for h in horizons]
            out = out.dropna(subset=delta_cols)
        else:
            mask = np.ones(len(out), dtype=bool)
            for h in horizons:
                future_close = out.groupby(date_col, sort=False)[close_col].shift(-h)
                mask &= future_close.notna().to_numpy()
            out = out.loc[mask].copy()

    return out



In [ ]:
delta_base_by_h = {60: delta_base_60, 90: delta_base_90}
delta_op_by_h   = {60: delta_op_60, 90: delta_op_90}
delta_tail_by_h = {60: delta_tail_60, 90: delta_tail_90}

In [ ]:
mnq_intraday_labeled = make_mnq_intraday_labeled(
    mnq_intraday,
    date_col="date",
    close_col="close",
    horizons=(60, 90),
    delta_base_by_h=delta_base_by_h,
    delta_op_by_h=delta_op_by_h,
    delta_tail_by_h=delta_tail_by_h,
    drop_na_targets=True,
    keep_delta=True,
)

In [ ]:
mnq_intraday_labeled

,date,open,high,low,close,volume,delta_60,delta_90
datetime,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,9.00,6.00
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,9.25,7.50
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,9.75,8.00
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,8.50,8.00
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,8.00,7.75
...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,21716.50,21722.75,21712.00,21719.50,1036,-90.00,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,21719.00,21719.75,21695.75,21698.50,3542,-69.25,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,21698.25,21700.25,21670.50,21679.25,5241,-52.25,-57.50


## **7. Análisis de `mnq_intraday_labeled`**


### **7.1. Objetivo del análisis:**

El objetivo de este análisis es caracterizar empíricamente el comportamiento del dataset `mnq_intraday_labeled` con foco en la dimensión temporal intradía, respondiendo, a partir de los datos, las siguientes preguntas clave:

- En qué minutos del día se concentra la mayor generación de señales
(`trade_60`, `trade_90`).
- Qué magnitud presentan los movimientos futuros asociados a dichas señales
(`delta_pts_60`, `delta_pts_90`).

- Qué tan económicamente relevantes son esas señales, evaluando la proporción de casos con $ \Delta pts_{t,h} \ge 50  \ $ puntos (u otros umbrales relevantes).

- Si existe persistencia temporal del movimiento, en particular:
  - cuándo y dónde $ |\Delta pts_{t,90}| $ tiende a ser mayor que $ |\Delta pts_{t,60}| $.

Este análisis permite identificar ventanas horarias intradía más activas y económicamente relevantes, y evaluar la consistencia temporal de los movimientos, como insumo para decisiones posteriores de filtrado, modelado y evaluación operativa.

### **7.2. Validaciones rápidas del dataset**

In [ ]:
mnq_intraday_labeled.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'delta_60',
       'delta_90'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np

df = mnq_intraday_labeled.copy()

#display(df.head())
#print(f'df.columns: {df.columns}\n')
#print(f'df.index.dtype: {df.index.dtype}\n')

# Validación: el índice debe ser DatetimeIndex
assert isinstance(df.index, pd.DatetimeIndex), "El índice debe ser DatetimeIndex (datetime)."

# Validación: columnas mínimas

required = [
    'delta_pts_60','trade_60', 'target_op_60', 'target_tail_60',
    'delta_pts_90','trade_90', 'target_op_90', 'target_tail_90'
                  ]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Faltan columnas: {missing}"

# Recuento básico de señales
print("Trades 60:", (df["trade_60"] != 0).sum())
print("Trades 90:", (df["trade_90"] != 0).sum())

Trades 60: 160670
Trades 90: 180453


### **7.3 Feature auxiliar: minuto del día (para agrupar intradía)**

In [ ]:
def add_minute_of_day (df):
  df["minute_of_day"] = df.index.hour * 60 + df.index.minute
  df["hour"] = df.index.hour
  df["minute"] = df.index.minute
  return df

mnq_intraday_labeled = add_minute_of_day(mnq_intraday_labeled)

### **7.4 Función principal: métricas por minuto del día**

Esta función calcula exactamente:

- Frecuencia: `n_trade`, `n_long`, `n_short` (para 60 y 90)
- Magnitud sobre `|delta_pts|` para trades: mean, p50, p75, p90 (para 60 y 90)
- Calidad: `%(|delta| >= thr_pts)` (para 60 y 90)
- Persistencia: `|delta_90| / |delta_60|` (stats + n_persist)

In [ ]:
import numpy as np
import pandas as pd

def metrics_by_minute_of_day(
    df: pd.DataFrame,
    thr_pts_by_h: dict = None,  # {60: 52.12, 90: 60.75}         # umbral mínimo en puntos para medir "calidad" del movimiento
    trade_60="trade_60",            # nombre columna trade para h=60
    trade_90="trade_90",            # nombre columna trade para h=90
    d60="delta_pts_60",             # nombre columna delta en puntos para h=60
    d90="delta_pts_90",             # nombre columna delta en puntos para h=90
) -> pd.DataFrame:
    """
    Calcula métricas agregadas por minuto del día (minute_of_day), incluyendo:
      - Frecuencia de señales (trades/long/short) para 60 y 90
      - Estadísticos de magnitud |delta| condicionados a trade != 0
      - Calidad: % de trades con |delta| >= thr_pts
      - Persistencia: ratio |delta_90| / |delta_60| cuando hay trade en ambos horizontes
    """

    # -------------------------
    # 1) Selección de columnas y features auxiliares
    # -------------------------
    # Copiamos solo las columnas necesarias para no tocar el df original
    dfx = df[[trade_60, trade_90, d60, d90, "minute_of_day"]].copy()

    # Magnitudes absolutas (nos interesan para evaluar "movimiento", sin dirección)
    dfx["abs60"] = dfx[d60].abs()
    dfx["abs90"] = dfx[d90].abs()

    # -------------------------
    # 2) Persistencia: |delta_90| / |delta_60|
    # -------------------------
    # Solo tiene sentido calcular ratio si:
    #  - hay señal (trade != 0) en ambos horizontes, y
    #  - abs60 > 0 para evitar división por cero
    mask_p = (dfx[trade_60] != 0) & (dfx[trade_90] != 0) & (dfx["abs60"] > 0)

    # Inicializamos la columna con NaN y solo llenamos donde corresponde
    dfx["persist_ratio"] = np.nan
    dfx.loc[mask_p, "persist_ratio"] = dfx.loc[mask_p, "abs90"] / dfx.loc[mask_p, "abs60"]

    # -------------------------
    # 3) Helpers: stats y porcentaje por umbral
    # -------------------------
    def stats(x: pd.Series) -> pd.Series:
        """
        Devuelve estadísticos robustos de una serie:
          mean, p50, p75, p90
        Si está vacía, devuelve NaN en todo.
        """
        x = x.dropna()
        if x.empty:
            return pd.Series({"mean": np.nan, "p50": np.nan, "p75": np.nan, "p90": np.nan})
        return pd.Series({
            "mean": float(x.mean()),
            "p50": float(x.quantile(0.50)),
            "p75": float(x.quantile(0.75)),
            "p90": float(x.quantile(0.90)),
        })

    def pct_ge(x: pd.Series, thr: float) -> float:
        """
        Porcentaje de valores >= umbral (thr).
        Si no hay datos, devuelve NaN.
        """
        x = x.dropna()
        return np.nan if x.empty else float((x >= thr).mean())

    # -------------------------
    # 4) GroupBy por minute_of_day
    # -------------------------
    # Agrupamos por minuto del día (0..1439)
    g = dfx.groupby("minute_of_day", sort=True)

    # DataFrame de salida indexado por minute_of_day
    out = pd.DataFrame(index=g.size().index)

    # -------------------------
    # 5) Frecuencia de señales (60)
    # -------------------------
    # trades totales: trade != 0
    out["n_trade_60"] = g.apply(lambda s: int((s[trade_60] != 0).sum()))
    # longs: trade == +1
    out["n_long_60"]  = g.apply(lambda s: int((s[trade_60] == 1).sum()))
    # shorts: trade == -1
    out["n_short_60"] = g.apply(lambda s: int((s[trade_60] == -1).sum()))

    # -------------------------
    # 6) Frecuencia de señales (90)
    # -------------------------
    out["n_trade_90"] = g.apply(lambda s: int((s[trade_90] != 0).sum()))
    out["n_long_90"]  = g.apply(lambda s: int((s[trade_90] == 1).sum()))
    out["n_short_90"] = g.apply(lambda s: int((s[trade_90] == -1).sum()))

    # -------------------------
    # 7) Magnitud |delta| condicionada a trade != 0
    # -------------------------
    # Para h=60: calculamos stats de abs60 solo en filas donde trade_60 != 0
    tmp60 = g.apply(lambda s: stats(s.loc[s[trade_60] != 0, "abs60"]))
    # Renombramos columnas para dejar claro qué representan
    tmp60.columns = [f"abs60_trade_{c}" for c in tmp60.columns]
    out = out.join(tmp60)

    # Para h=90: stats de abs90 solo en filas donde trade_90 != 0
    tmp90 = g.apply(lambda s: stats(s.loc[s[trade_90] != 0, "abs90"]))
    tmp90.columns = [f"abs90_trade_{c}" for c in tmp90.columns]
    out = out.join(tmp90)

    # -------------------------
    # 8) Calidad: % de trades con |delta| >= thr_pts
    # -------------------------
    # Esto mide cuán "económicas" son las señales según un umbral simple en puntos
    out["pct_abs60_ge_thr_trade"] = g.apply(
        lambda s: pct_ge(s.loc[s[trade_60] != 0, "abs60"], thr_pts_by_h[60])
    )

    out["pct_abs90_ge_thr_trade"] = g.apply(
        lambda s: pct_ge(s.loc[s[trade_90] != 0, "abs90"], thr_pts_by_h[90])
    )

    # -------------------------
    # 9) Persistencia: cantidad y estadísticos del ratio |90|/|60|
    # -------------------------
    # n_persist: cuántas observaciones del grupo tienen persist_ratio definido
    out["n_persist"] = g.apply(lambda s: int(s["persist_ratio"].notna().sum()))

    # stats del ratio de persistencia
    tmpp = g.apply(lambda s: stats(s["persist_ratio"]))
    tmpp.columns = [f"persist_ratio_{c}" for c in tmpp.columns]
    out = out.join(tmpp)

    # -------------------------
    # 10) Campos de hora/minuto legibles
    # -------------------------
    # minute_of_day: 0..1439 => hour = //60, minute = %60
    out["hour"] = out.index // 60
    out["minute"] = out.index % 60

    # Reordenamos columnas para que hour/minute queden al inicio
    out = out[["hour", "minute"] + [c for c in out.columns if c not in ("hour", "minute")]]

    return out


### **7.5. Ejecutar el análisis**

In [ ]:
thr_pts_by_h = {
    60: delta_base_60,   # delta_base_60
    90: delta_base_90,   # delta_base_90
}

minute_metrics = metrics_by_minute_of_day(
    mnq_intraday_labeled,
    thr_pts_by_h=thr_pts_by_h
)

minute_metrics.head()

,hour,minute,n_trade_60,n_long_60,n_short_60,n_trade_90,n_long_90,n_short_90,abs60_trade_mean,abs60_trade_p50,...,abs90_trade_p50,abs90_trade_p75,abs90_trade_p90,pct_abs60_ge_thr_trade,pct_abs90_ge_thr_trade,n_persist,persist_ratio_mean,persist_ratio_p50,persist_ratio_p75,persist_ratio_p90
minute_of_day,,,,,,,,,,,,,,,,,,,,,
390,6,30,104,45,59,114,45,69,75.052885,68.500,...,76.50,103.4375,151.125,1.0,1.0,65,1.338743,1.160000,1.394144,1.891755
391,6,31,109,42,67,113,43,70,74.743119,68.000,...,78.00,97.7500,150.900,1.0,1.0,70,1.328297,1.148019,1.358230,1.980020
392,6,32,109,42,67,114,42,72,73.555046,68.250,...,76.75,93.2500,144.125,1.0,1.0,66,1.342719,1.206316,1.376303,1.940112
393,6,33,92,38,54,106,42,64,76.809783,69.625,...,77.00,99.5000,140.750,1.0,1.0,54,1.250238,1.166732,1.271257,1.677677
394,6,34,93,40,53,108,42,66,76.338710,69.750,...,76.50,95.8125,136.200,1.0,1.0,54,1.316143,1.212749,1.339015,1.719118


### **7.6 Tablas “Top” para interpretar rápido**

#### 7.6.1. Top 20 por frecuencia (60 min)

La tabla muestra los 20 minutos del día más activos para H=60, indicando cuántas señales se generan, qué magnitud típica tienen y qué proporción supera el umbral base económico.

In [ ]:
top60 = minute_metrics.sort_values("n_trade_60", ascending=False).head(20)
display(top60[["hour","minute","n_trade_60","abs60_trade_mean","abs60_trade_p50","abs60_trade_p90","pct_abs60_ge_thr_trade"]])

,hour,minute,n_trade_60,abs60_trade_mean,abs60_trade_p50,abs60_trade_p90,pct_abs60_ge_thr_trade
minute_of_day,,,,,,,
570,9,30,687,108.812591,95.500,172.500,1.0
568,9,28,679,109.101620,96.250,173.750,1.0
569,9,29,676,109.875000,96.500,178.875,1.0
563,9,23,668,108.366392,93.875,178.500,1.0
571,9,31,666,108.991366,95.000,178.125,1.0
565,9,25,654,110.540520,96.250,180.425,1.0
564,9,24,653,109.489280,95.500,179.950,1.0
567,9,27,651,110.871736,97.000,174.500,1.0
572,9,32,647,108.056801,93.750,173.450,1.0


**Comentarios:**

- Las señales para el horizonte de 60 minutos se concentran casi exclusivamente en la franja 09:18–09:41, con un pico marcado entre 09:25 y 09:32, lo que evidencia una ventana intradía estructuralmente activa.

- En estos minutos, los movimientos asociados presentan una magnitud típica elevada (mediana ≈ 92–97 pts, P90 ≈ 170–180 pts), superando ampliamente el delta base y acercándose al delta operativo.

- Además, el 100 % de las señales en este bloque supera el umbral económico mínimo, lo que indica alta frecuencia con calidad económica.

- En conjunto, estos resultados confirman que el horizonte H = 60 min es altamente dependiente del timing, y que su uso operativo o modelado se beneficia claramente de un filtro horario explícito.

#### 7.6.2. Top 20 por frecuencia (90 min)

In [ ]:
top90 = minute_metrics.sort_values("n_trade_90", ascending=False).head(20)
display(top90[["hour","minute","n_trade_90","abs90_trade_mean","abs90_trade_p50","abs90_trade_p90","pct_abs90_ge_thr_trade"]])


,hour,minute,n_trade_90,abs90_trade_mean,abs90_trade_p50,abs90_trade_p90,pct_abs90_ge_thr_trade
minute_of_day,,,,,,,
562,9,22,671,124.144188,105.250,202.000,1.0
560,9,20,670,124.693284,107.125,201.800,1.0
561,9,21,668,124.701722,106.000,207.950,1.0
564,9,24,663,126.361614,108.500,208.900,1.0
558,9,18,661,123.639183,107.750,199.500,1.0
559,9,19,661,124.343797,107.750,202.500,1.0
570,9,30,661,126.686838,108.750,203.500,1.0
569,9,29,659,126.475341,109.250,200.600,1.0
568,9,28,659,126.575114,109.000,200.200,1.0


**Comentarios:**

- Las señales para el horizonte de 90 minutos se concentran principalmente en la franja 09:08–09:31, mostrando una ventana intradía más amplia que la observada para H = 60.

- En estos minutos, los movimientos asociados presentan una magnitud típica mayor (mediana ≈ 105–111 pts, P90 ≈ 198–209 pts), coherente con la mayor duración del horizonte y con el delta operativo de H = 90.

- Al igual que en H = 60, el 100 % de las señales dentro de este Top 20 supera el umbral económico base, lo que confirma una alta calidad económica junto con una frecuencia elevada.

- En conjunto, estos resultados indican que el horizonte H = 90 min ofrece una mayor tolerancia temporal y una estabilidad superior, reforzando su elección como horizonte operativo principal.


#### 7.6.3. Top 20 por persistencia (90/60)

El siguiente código ordena los minutos del día según el valor medio de persistencia (`persist_ratio_mean = |Δ90| / |Δ60|`).

- Selecciona los 20 minutos en los que, en promedio, el movimiento a 90 minutos es significativamente mayor que el movimiento a 60 minutos.

- Para dichos minutos se reporta:
  - `n_persist`: cuántas veces ocurrió esa comparación válida,
  - `persist_ratio_mean`: ratio promedio |Δ90| / |Δ60|,
  - `persist_ratio_p50`: ratio típico (mediana),
  - `persist_ratio_p90`: escenarios de persistencia fuerte.

En síntesis, este análisis identifica ventanas intradía donde el movimiento no solo se produce, sino que tiende a continuar y ampliarse al extender el horizonte temporal de 60 a 90 minutos.

La persistencia no constituye una señal de entrada, sino un indicador de continuidad, útil para:
- decidir cuándo mantener una operación,
- evaluar la viabilidad de targets de mayor magnitud,
- y diseñar una operatoria intradía más eficiente y menos reactiva.

En resumen: **no es señal de entrada, es señal de continuidad.**

**¿Por qué se comparan los horizontes de 60 y 90 minutos?**

La comparación entre H = 60 y H = 90 permite distinguir entre:
- impulso inicial (movimiento que aparece rápido pero se agota),
- y expansión real del movimiento (movimiento que continúa y se amplía).

Si $∣Δ90∣>∣Δ60∣$ , el mercado muestra follow-through, lo que indica que extender el horizonte agrega valor económico.

Por el contrario, si el ratio es cercano a 1, el movimiento tiende a saturarse temprano.

Esta comparación transforma la dimensión temporal en un criterio operativo explícito para la gestión de posiciones.

In [ ]:
topPersist = minute_metrics.sort_values("persist_ratio_mean", ascending=False).head(25)
display(topPersist[["hour","minute","n_persist","persist_ratio_mean","persist_ratio_p50","persist_ratio_p90"]])

,hour,minute,n_persist,persist_ratio_mean,persist_ratio_p50,persist_ratio_p90
minute_of_day,,,,,,
511,8,31,158,1.706157,1.549879,2.721111
514,8,34,196,1.667025,1.577474,2.594541
512,8,32,183,1.641805,1.491667,2.582157
515,8,35,197,1.632405,1.492877,2.510926
513,8,33,194,1.621595,1.500090,2.526770
516,8,36,220,1.589260,1.470331,2.426089
517,8,37,239,1.573854,1.449782,2.400662
528,8,48,318,1.568026,1.406429,2.375474
522,8,42,273,1.556397,1.447368,2.283163


In [ ]:
gtn_index_min = topPersist.index.min()
gtn_index_max = topPersist.index.max()

gestation_window_start = f"{gtn_index_min//60:02d}:{gtn_index_min%60:02d}"
gestation_window_end  = f"{gtn_index_max//60:02d}:{gtn_index_max%60:02d}"

gestation_window_start, gestation_window_end

('08:21', '08:49')

**Comentarios:**

- Desplazamiento temporal claro:

  La mayor persistencia se concentra entre 08:21 y 08:48, antes del bloque de mayor frecuencia y magnitud observado en 09:xx.

- Persistencia > 1 de forma sistemática:

  `persist_ratio_mean ≈ 1.50–1.71`, lo que indica que el movimiento a 90 min tiende a extender consistentemente al de 60 min (follow-through intradía).

- Colas relevantes:

  `persist_ratio_p90 ≈ 2.15–2.72`, evidenciando episodios donde el movimiento a 90 minutos duplica o más al de 60.

- Soporte estadístico adecuado:

  `n_persist ≈ 145–318` por minuto, suficiente para considerar el patrón estable y no marginal.

- Lectura operativa:

  Este bloque es óptimo para detección temprana y evaluación de continuidad, mientras que el bloque 09:xx resulta más adecuado para explotación de magnitud.

- Implicación directa — estrategia en dos fases:
  - 08:xx → señalización y persistencia,
  - 09:xx → monetización y amplitud.

### **7.7 Interpretación mínima de cada métrica (para el informe)**


- `n_trade_60`, `n_trade_90`: cuántas veces el modelo “decidió operar” en ese minuto del día a lo largo del histórico.
- `abs*_trade_p50`: movimiento “típico” (mediana) en puntos.
- `abs*_trade_p90`: magnitud en escenarios fuertes (top 10%).
- `pct_abs*_ge_thr_trade`: proporción de señales “económicamente relevantes” (ej. ≥ 25 pts).
- `persist_ratio_*`: si es > 1, suele indicar que el movimiento a 90 min tiende a ser mayor que a 60 min (extensión).

## **8. Construcción y justificación de la ventana operativa**

A partir del análisis intradía por minuto del día —basado en métricas de frecuencia de señales, magnitud del movimiento, calidad económica y persistencia temporal (60 → 90 min)— se observa que el comportamiento del mercado no es homogéneo a lo largo de la sesión, sino que presenta regímenes temporales diferenciados.

El análisis empírico muestra, por un lado, minutos tempranos caracterizados por alta persistencia del movimiento, donde los desplazamientos tienden a continuarse y ampliarse al extender el horizonte temporal. Por otro lado, se identifican minutos posteriores con mayor frecuencia y magnitud, donde se concentran los movimientos de mayor amplitud económica.

Esta evidencia justifica el paso metodológico desde un análisis minuto a minuto hacia la definición de bloques horarios, no como una simplificación arbitraria, sino como una abstracción natural del comportamiento observado en los datos.

En este contexto, resulta adecuado conceptualizar dos ventanas operativas diferenciadas:

- una ventana de gestación, asociada a detección temprana y continuidad del movimiento,
- y una ventana de expansión, asociada a explotación de magnitud y monetización.

Esta distinción permite alinear el análisis estadístico con la lógica operativa intradía, habilitando decisiones diferenciadas de entrada, gestión y extensión de las operaciones, y estableciendo una base sólida para la definición final de la ventana operativa.

### **8.1 Ventana de gestación (predicción)**

El ranking de persistencia (`persist_ratio_mean`) muestra valores elevados de forma consistente en la franja 08:21–08:48, indicando que, en estos minutos, los movimientos observados a 60 minutos tienden a extenderse y amplificarse en horizontes posteriores (90 minutos).

Esta evidencia sugiere que:
- en esta franja el mercado define dirección y estructura el impulso, aunque aún no alcanza su máxima magnitud,
- la información contenida en este período es anticipatoria, ya que precede sistemáticamente a los movimientos más relevantes del día,
- los patrones observados presentan alta continuidad temporal, lo que los hace adecuados para inferencia predictiva.

Por este motivo, esta franja se define como ventana de gestación o predicción, y se utiliza para:

- construir las variables de entrada (features) del modelo,
- capturar señales tempranas del impulso intradía,
- entrenar el modelo con el objetivo de anticipar la ocurrencia y magnitud de los movimientos que se materializan posteriormente en la ventana de expansión.

En este esquema, la ventana de gestación aporta la información predictiva, mientras que la ventana de expansión constituye el horizonte objetivo sobre el cual se evalúa la validez y utilidad operativa de dichas predicciones.

### **8.2 Ventana de expansión (ejecución)**

El ranking por frecuencia operativa (`n_trade_60`, `n_trade_90`) y magnitud del movimiento (`abs*_trade_p50`, `abs*_trade_p90`) muestra máximos claros y recurrentes en la franja 09:00–10:00, caracterizada por:

- Alta concentración de señales por minuto.
- Movimientos típicos y extremos significativamente superiores a los umbrales económicos definidos.
- Proporción cercana al 100 % de operaciones con $ ∣Δ𝑝𝑡𝑠∣≥Δ_{𝑏𝑎𝑠𝑒}$.

Esta evidencia indica que, en este bloque horario:
- el movimiento ya se encuentra activo y extendido,
- la señal deja de ser anticipatoria y pasa a ser económicamente explotable,
- el edge observado es operativo, no solo estadísticamente significativo.

En consecuencia, esta franja se define como ventana de expansión o ejecución, es decir, el período en el cual las señales anticipadas durante la ventana de gestación pueden materializarse en operaciones reales, orientadas a la monetización del movimiento intradía.

### **8.3. Justificación del desfase temporal entre predicción y ejecución**


Un resultado central de este análisis es la existencia de un desfase temporal consistente entre:
- La franja donde se maximiza la persistencia (gestación del movimiento).
- La franja donde se maximiza la frecuencia y magnitud (expansión del movimiento).

Este desfase justifica el criterio metodológico de:

- Entrenar el modelo con datos comprendidos entre 08:00 y 09:00, y
- Utilizar sus predicciones para operar en el período 09:00–10:00.

Dicho enfoque evita el uso de información contemporánea o futura (data leakage) y alinea el diseño del modelo con la estructura temporal observada empíricamente en los datos.

### **8.4. Conclusión del punto 8**

El análisis intradía de `mnq_intraday_labeled` permite concluir que la definición de ventanas operativas no es arbitraria, sino que se encuentra respaldada por métricas objetivas extraídas del propio dataset. La separación entre una ventana de predicción y una ventana de ejecución constituye, por lo tanto, una decisión metodológica coherente con la dinámica observada del mercado y sienta las bases para el diseño experimental de las etapas posteriores del pipeline

La secuencia temporal correcta sería:

1. Ventana de gestación (08:20–08:40)
    - Se construyen las features.
    - El modelo predice si se alcanzará un determinado delta_* (base / operativo / extensión) a 90 minutos.

2. Ventana de expansión (09:10–09:40)
    - Se toma la operación solo si la predicción es favorable.
    - El punto de entrada es algún minuto dentro de esta ventana.

3. Horizonte de resultado (t + 90 min desde la entrada)
    - La operación se gestiona y se cierra según:
      - `delta_op` (objetivo),
      - `delta_tail` (extensión),
      - o reglas de stop.



**Aclaración clave (muy importante)**
- El modelo NO predice “qué pasa dentro de la ventana de expansión”.
- El modelo predice qué va a pasar DESPUÉS, 90 minutos desde la entrada.

La ventana de expansión es:
- El momento de ejecución.
- No el horizonte del target.

La ventana de gestación (08:20–08:40) se utiliza para generar predicciones sobre el movimiento futuro del mercado a un horizonte de 90 minutos. Dichas predicciones se materializan operativamente mediante entradas realizadas durante la ventana de expansión (09:10–09:40), evaluándose el resultado económico a partir de cada punto de entrada hacia 90 minutos adelante.

### **8.5 Uso del `dataset mnq_intraday_labeled` bajo el criterio de ventanas operativas**

Bajo el criterio de ventanas operativas definido en los apartados anteriores, el dataset `mnq_intraday_labeled` se mantiene como la fuente completa de información intradía, sin modificaciones en su estructura original.

A partir de este dataset se construye un conjunto de datos derivado para entrenamiento y evaluación del modelo, aplicando el siguiente esquema día a día:

**Variables de entrada (features)**

Las variables de entrada se extraen exclusivamente de las observaciones comprendidas dentro de la ventana de gestación del movimiento (08:20–08:40).

Estas variables incluyen precios OHLCV y atributos técnicos derivados, y representan toda la información disponible antes de la activación y expansión del movimiento intradía.

**Variables objetivo (targets)**

Las variables objetivo se obtienen a partir de las columnas ya calculadas en `mnq_intraday_labeled`, evaluadas a un horizonte fijo de 90 minutos desde cada punto de entrada seleccionado dentro de la ventana de expansión (09:10–09:40).

En particular, los targets corresponden a los deltas a 90 minutos (delta_base_90, delta_op_90, delta_tail_90, o sus versiones discretizadas), que representan la materialización económica del movimiento.

**Separación temporal y validez metodológica**

Este diseño garantiza una separación temporal estricta entre:
- información utilizada como entrada (ventana de gestación),
- y resultados evaluados a futuro (90 minutos desde la entrada),

evitando el uso de información futura (data leakage) y alineando el entrenamiento del modelo con la estructura temporal observada empíricamente en los datos y con la lógica operativa real.

**El modelo utiliza información temprana para anticipar movimientos que se materializan más adelante, respetando una separación temporal clara entre predicción, ejecución y resultado.**

## **9. Bloque de código único, modular y reutilizable**

In [31]:
mnq_intraday

,date,open,high,low,close,volume
datetime,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3
...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859


In [38]:
import numpy as np
import pandas as pd

def compute_targets_returns(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    horizons: tuple[int, ...] = (60, 90),
) -> pd.DataFrame:
    """
    Fórmulas:
      r_{t,h}  = (P_{t+h} - P_t) / P_t
    Sin cruzar días (shift por date). Índice datetime intacto.
    """
    out = df.copy()

    for h in horizons:
        P_t = out[close_col]
        P_th = out.groupby(date_col, group_keys=False)[close_col].shift(-h)

        out[f"ret_{h}"]   = (P_th - P_t) / P_t

    return out

In [40]:
import numpy as np
import pandas as pd

def validate_targets_no_cross_day(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    horizons: tuple[int, ...] = (60, 90),
    rtol: float = 1e-10,
    atol: float = 1e-12,
) -> None:
    """
    Comprueba:
      1) ret_h == (P_{t+h}-P_t)/P_t  (y equivalencia con P_{t+h}/P_t - 1)
      2) No cruza días: donde exista P_{t+h}, su 'date' coincide con date actual
      3) NaNs por día: en cada día, los últimos h registros tienen NaN (exactamente h)

    Lanza AssertionError si algo falla.
    """
    if close_col not in df.columns or date_col not in df.columns:
        raise ValueError(f"Faltan columnas: requiere '{close_col}' y '{date_col}'")

    for h in horizons:
        ret_col = f"ret_{h}"
        if ret_col not in df.columns:
            raise ValueError(f"Falta la columna requerida: '{ret_col}'")

        P_t  = df[close_col]
        P_th = df.groupby(date_col, group_keys=False)[close_col].shift(-h)

        # -----------------------------
        # 1) Verificación fórmulas ret
        # -----------------------------
        expected_ret  = (P_th - P_t) / P_t
        expected_ret2 = (P_th / P_t) - 1.0

        # comparar solo donde hay datos (no NaN)
        m = P_th.notna() & P_t.notna()

        assert np.allclose(
            df.loc[m, ret_col].to_numpy(),
            expected_ret.loc[m].to_numpy(),
            rtol=rtol,
            atol=atol,
        ), f"{ret_col} no coincide con (P_th - P_t)/P_t"

        # equivalencia algebraica
        assert np.allclose(
            expected_ret.loc[m].to_numpy(),
            expected_ret2.loc[m].to_numpy(),
            rtol=rtol,
            atol=atol,
        ), f"{ret_col} no es equivalente a P_th/P_t - 1"

        # -----------------------------
        # 2) No cruza días (verifica date de t+h)
        # -----------------------------
        date_t  = df[date_col]
        date_th = df.groupby(date_col, group_keys=False)[date_col].shift(-h)

        m_date = date_th.notna()
        assert (date_th.loc[m_date] == date_t.loc[m_date]).all(), (
            f"Cruce de día detectado en h={h}: date(t+h) != date(t)"
        )

        # -----------------------------
        # 3) NaNs solo en últimos h de cada día (exactamente h)
        # -----------------------------
        nan_counts = df[ret_col].isna().groupby(df[date_col]).sum()
        sizes = df.groupby(date_col).size()
        check_days = sizes[sizes >= h].index

        assert (nan_counts.loc[check_days] == h).all(), (
            f"Conteo de NaNs por día incorrecto para h={h}. "
            f"Se esperaba h NaNs por día (en días con >=h filas)."
        )

    print("Validación OK: ret, no cruce de días y NaNs por jornada/horizonte.")


In [41]:
mnq_intraday_targets = compute_targets_returns(
    mnq_intraday,
    close_col="close",
    date_col="date",
    horizons=(60, 90),
)

validate_targets_no_cross_day(mnq_intraday_targets)

Validación OK: ret, no cruce de días y NaNs por jornada/horizonte.


In [43]:
OUT_PARQUET

PosixPath('/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet')

In [44]:
from pathlib import PosixPath

output_path = PosixPath(
    "/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet"
)

# Asegurar que el directorio exista
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Guardar en formato parquet
mnq_intraday_targets.to_parquet(OUT_PARQUET, index=True)